# Chapter 16: Simultaneous Equations Models

Topics: setup and notation, estimation by 2SLS, overview of 3SLS.

## Step 1 – Setup: Working Directory & Output Folder

Run this cell first to fix the working directory and create the `output/` folder.

In [ ]:
import numpy as np
import pandas as pd
import wooldridge as woo
import statsmodels.formula.api as smf
import os
from linearmodels.iv import IV2SLS

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

## 16.1 Supply and Demand Example

## Step 2 – Simultaneous Equations Models

In a **system of simultaneous equations**, variables are jointly determined. OLS on any single equation is biased due to **simultaneity bias**.

**Labour market example (structural form):**
- Labour supply: $\log(h^s) = \alpha_1 \log(w) + \beta_{10} + \beta_{11}z_1 + u_1$
- Labour demand: $\log(h^d) = \alpha_2 \log(w) + \beta_{20} + \beta_{21}z_2 + u_2$

Equilibrium: $h^s = h^d = h$

$z_1$, $z_2$ are **excluded exogenous variables** that shift supply and demand respectively.

In [ ]:
df = woo.dataWoo('mroz').dropna()

# Structural equation 1: labour supply
supply = IV2SLS.from_formula(
    'hours ~ 1 + lwage + kidslt6 + nwifeinc + [lwage ~ educ + exper + expersq]',
    data=df
).fit()
print("Labour Supply (2SLS):")
print(supply.summary)

## 16.2 Reduced Form and Identification

## Step 3 – Identification Conditions

**Order condition** (necessary for identification):
$$M - m \geq g - 1$$

where:
- $M$ = total number of excluded exogenous variables in the system
- $m$ = number of exogenous variables included in the equation
- $g$ = number of endogenous variables in the equation

If $M - m = g - 1$: exactly identified
If $M - m > g - 1$: overidentified (use 2SLS)
If $M - m < g - 1$: underidentified (cannot estimate consistently)

**Reduced form:** express each endogenous variable as a function of all exogenous variables only.

In [ ]:
# Reduced form: regress endogenous on all exogenous
rf_lwage = smf.ols('lwage ~ educ + exper + expersq + kidslt6 + nwifeinc', data=df).fit()
print("Reduced Form for lwage:")
print(rf_lwage.summary())
print(f"\nF-stat (instrument relevance): {rf_lwage.fvalue:.2f}")

## Step 4 – 2SLS Estimation of Structural Equations

**Estimation strategy:**

1. Write the **reduced form** for each endogenous RHS variable
2. Use exogenous variables as instruments
3. Apply **2SLS** equation by equation

Using `linearmodels.iv.IV2SLS` in Python:
```python
from linearmodels.iv import IV2SLS
res = IV2SLS.from_formula(
    'y ~ 1 + exog + [endog ~ instrument]', data=df
).fit()
```

**Check instrument strength** via first-stage F-statistic > 10.